# Refua Antibody Design Notebook: HER2 Domain IV

> **Audience:** anyone new to antibody design workflows.  
> **Goal:** generate epitope-focused antibody candidates against a clinically important HER2 surface region.

## Why this target is interesting (with sources)
- **Biology and disease relevance:** HER2 is implicated in aggressive cancers; classic HER2 structural literature reports overexpression in 20-30% of breast cancers and links it to poorer prognosis.
- **Clinical signal remains current:** on **November 20, 2024**, FDA granted accelerated approval to **zanidatamab-hrii**, a HER2-directed bispecific antibody, for previously treated unresectable/metastatic HER2-positive biliary tract cancer.
- **Structure-guided design is practical:** PDB **1N8Z** resolves human HER2 extracellular domain in complex with Herceptin (trastuzumab) Fab, which supports domain- and epitope-focused design setups.

## Workflow
1. Define a HER2 domain IV antigen and a trastuzumab-facing window.
2. Generate heavy/light antibody templates with explicit CDR length controls.
3. Run one design/fold pass and inspect candidate specs for triage.


In [ ]:
%load_ext refua_notebook
import refua_notebook

refua_notebook.activate()


In [ ]:
from refua import BinderDesigns, Complex, Protein


## Biological setup (plain language)

This notebook uses a HER2 domain IV segment from PDB 1N8Z (chain C, juxtamembrane region), where trastuzumab-class antibodies bind.

| Design input | Choice in this notebook | Why |
|---|---|---|
| Target chain | 1N8Z HER2 domain IV segment (119 aa) | Compact structural region that includes the Herceptin-facing surface |
| Binding hotspot | `38..110` | Broadly spans the central trastuzumab-contact neighborhood for focused exploration |
| Candidate style | De novo heavy/light pair with CDR constraints | Standard early-stage antibody exploration pattern |


In [ ]:
HER2_DOMAIN_IV_1N8Z = (
    "CHQLCARGHCWGPGPTQCVNCSQFLRGQECVEECRVLQGLPREYVNARHCLPCHPECQPQNGSVTCFGPEADQCVACAHYKDPPFCVARCPSGVKPDLSYMPIWKFPDEEGACQPCPIN"
)

# Reference therapeutic binder sequences from PDB 1N8Z (trastuzumab / Herceptin Fab).
TRASTUZUMAB_HEAVY_1N8Z = (
    "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKVEP"
)
TRASTUZUMAB_LIGHT_1N8Z = (
    "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIKRTVAAPSVFIFPPSDEQLKSGTASVVCLLNNFYPREAKVQWKVDNALQSGNSQESVTEQDSKDSTYSLSSTLTLSKADYEKHKVYACEVTHQGLSSPVTKSFNRGEC"
)

HER2_BINDING_WINDOW = "38..110"

HEAVY_CDR_LENGTHS = (12, 10, 13)
LIGHT_CDR_LENGTHS = (10, 9, 9)


## Build antibody design spec

This cell composes:
- the antigen (`A`)
- a generated heavy chain (`H`)
- a generated light chain (`L`)

into one `Complex` object named `her2_domain4_antibody_design`.

Think of `design_spec` as a reusable recipe you can rerun across nearby windows and alternate CDR settings.


In [ ]:
antigen = Protein(
    HER2_DOMAIN_IV_1N8Z,
    ids="A",
    binding_types={"binding": HER2_BINDING_WINDOW},
)

binder_pair = BinderDesigns.antibody(
    heavy_cdr_lengths=HEAVY_CDR_LENGTHS,
    light_cdr_lengths=LIGHT_CDR_LENGTHS,
    heavy_id="H",
    light_id="L",
)

design_spec = Complex([antigen, *binder_pair], name="her2_domain4_antibody_design")


In [ ]:
{
    "heavy_spec": binder_pair.heavy.sequence,
    "light_spec": binder_pair.light.sequence,
    "trastuzumab_ref_heavy_1n8z": TRASTUZUMAB_HEAVY_1N8Z,
    "trastuzumab_ref_light_1n8z": TRASTUZUMAB_LIGHT_1N8Z,
}


## Design narrative: from intent to candidates

This is an **early discovery pass** to generate structured antibody hypotheses for ranking and follow-up.

### What each output means
| Output | What it is | How to use it |
|---|---|---|
| `result.binder_specs` | Heavy/light candidate specifications | Compare sequence hypotheses across CDR settings |
| `result.chain_design_summary()` | Per-chain summary of fixed vs designable regions | Check whether constraints match your intent |
| `result.features` | Model-level features for downstream ranking workflows | Save for reproducibility and triage pipelines |

> Practical strategy: run multiple nearby hotspot windows and keep candidates that remain stable across settings.


## Run a design pass

`design_spec.fold()` executes the design/fold workflow for this setup.

Interpretation guardrails:
- Generated binders are **computational candidates**, not validated therapeutics.
- Use these outputs for prioritization and shortlist generation.
- Confirm top candidates experimentally (binding, function, developability, safety).


In [ ]:
result = design_spec.fold()

result


In [ ]:
result.binder_specs


In [ ]:
result.chain_design_summary()


## Science references

- FDA (November 20, 2024), accelerated approval of HER2-directed bispecific antibody zanidatamab-hrii in HER2+ BTC: https://www.fda.gov/drugs/resources-information-approved-drugs/fda-grants-accelerated-approval-zanidatamab-hrii-previously-treated-unresectable-or-metastatic-her2
- RCSB PDB 1N8Z (human HER2 extracellular domain in complex with Herceptin Fab): https://www.rcsb.org/structure/1N8Z
- RCSB FASTA 1N8Z (exact chain sequences): https://www.rcsb.org/fasta/entry/1N8Z/display
- Cho et al., *Nature* (2003), HER2 extracellular structure + Herceptin Fab (PMID: 12610629): https://pubmed.ncbi.nlm.nih.gov/12610629/
- FDA preclinical research basics (why computational designs require experimental validation): https://www.fda.gov/patients/drug-development-process/step-2-preclinical-research
